# Recuperación — corpus Sistema Penal Acusatorio

Este notebook mide **recuperación** sobre los chunks producidos por la librería
`shared.legal_chunking` (la misma que valida el notebook de chunking). Compara tres
recuperadores sobre los MISMOS chunks:

* **denso** — coseno sobre embeddings (Qwen3-Embedding vía TEI),
* **BM25** — léxico (`shared.lexical`),
* **híbrido** — fusión de ambos con RRF (Reciprocal Rank Fusion).

Hallazgo del notebook de chunking: con preguntas parafraseadas el denso solo trae el
*vecino temático*, no el artículo exacto. La hipótesis aquí es que el componente
léxico (BM25) rescata esos casos, porque la pregunta comparte términos jurídicos con
la respuesta aunque no la frase. Métrica: `recall@k` y `MRR` (k amplio: los rerankers
operarían sobre este top-N). Requiere el túnel al TEI arriba (`./tunnel.sh`).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

current_dir = Path.cwd()
labs_folder = next(d for d in [current_dir, *current_dir.parents] if (d / 'shared').exists())
if str(labs_folder) not in sys.path:
    sys.path.insert(0, str(labs_folder))

from shared import legal_chunking as lc
from shared.lexical import BM25, tokenize, rank_indices_by_score, rrf
from shared.tei_client import TEIClient

CORPUS_PATH = labs_folder / 'ingestion' / 'out' / 'Sistema Penal Acusatorio'
CACHE_DIR = labs_folder / 'exploracion_datos' / '.embed_cache'   # comparte caché con el otro notebook
print('Corpus:', CORPUS_PATH)

Corpus: /Users/savashito/Claude/Projects/AI_research/RAG/labs/ingestion/out/Sistema Penal Acusatorio


## 1. Chunks desde la librería

Sin redefinir nada: se limpia el corpus y se arma el chunking por estructura con
`shared.legal_chunking`, y un baseline de tamaño fijo sobre el mismo texto limpio.

In [2]:
raw = lc.read_markdown_dir(CORPUS_PATH)
documents = lc.clean_corpus(raw)
chunks = lc.chunk_documents(documents)      # estructura (art/heading + merge + partición)

def fixed_size_chunks(text, size=lc.TARGET_WORDS, overlap=lc.OVERLAP_WORDS):
    w = text.split(); step = max(1, size - overlap); out = []
    for s in range(0, len(w), step):
        piece = w[s:s + size]
        if piece:
            out.append(' '.join(piece))
        if s + size >= len(w):
            break
    return out

baseline = pd.DataFrame([{'source': name, 'text': c}
                         for name, text in documents.items()
                         for c in fixed_size_chunks(text)])
print(f'docs: {len(documents)} · chunks estructura: {len(chunks)} · chunks baseline: {len(baseline)}')
chunks[['source', 'unit_type', 'title', 'words']].head()

docs: 24 · chunks estructura: 3616 · chunks baseline: 3650


,source,unit_type,title,words
0,Aplicación del CNPP.md,heading,Capítulo segundo,376
1,Aplicación del CNPP.md,heading,PRESENTACIÓN,234
2,Aplicación del CNPP.md,heading,LA NECESIDAD DE UN PROCEDIMIENTO PENAL ÚNICO P...,796
3,Aplicación del CNPP.md,heading,LA NECESIDAD DE UN PROCEDIMIENTO PENAL ÚNICO P...,772
4,Aplicación del CNPP.md,heading,LA NECESIDAD DE UN PROCEDIMIENTO PENAL ÚNICO P...,785


## 2. Preguntas difíciles y embeddings

Preguntas parafraseadas / de escenario (mínimo solape léxico con la respuesta),
ancladas a una frase verificable que el chunk correcto contiene. Los embeddings de
chunks salen de la caché compartida; sólo las consultas se embeben en vivo.

In [3]:
golden = pd.DataFrame([
    {'q': '¿Se puede tratar a un acusado como culpable antes de que un juez dicte sentencia?',
     'answer': 'se presume inocente y será tratada como tal'},
    {'q': '¿Qué grado de certeza necesita el tribunal para poder condenar a una persona?',
     'answer': 'más allá de toda duda razonable'},
    {'q': '¿Cómo se garantiza que la evidencia recogida en la escena no se altere ni se cambie?',
     'answer': 'sistema de control y registro que se aplica al indicio'},
    {'q': 'Si la policía obtiene una prueba violando derechos, ¿esa prueba puede usarse en el juicio?',
     'answer': 'Cualquier acto realizado con violación de derechos humanos será nulo'},
    {'q': '¿Qué debe ocurrir para que el fiscal logre que el proceso continúe formalmente contra el imputado?',
     'answer': 'dictará el auto de vinculación del imputado a proceso'},
    {'q': '¿Puede la víctima llegar a un arreglo con el acusado para resolver el asunto sin llegar a juicio?',
     'answer': 'acuerdos reparatorios son aquéllos celebrados entre la víctima u ofendido y el imputado'},
    {'q': '¿Pueden juzgar a alguien otra vez por un delito del que ya fue absuelto?',
     'answer': 'no podrá ser sometida a otro proceso penal por los mismos hechos'},
    {'q': '¿La defensa tiene derecho a rebatir y objetar los elementos que presenta la fiscalía?',
     'answer': 'controvertir o confrontar los medios de prueba'},
    {'q': '¿En qué condición es legítimo detener y retener físicamente a una persona?',
     'answer': 'nadie podrá ser privado de la misma'},
    {'q': '¿Qué diligencias puede llevar a cabo el ministerio público sin pedir permiso a un juez?',
     'answer': 'No requieren autorización del Juez de control'},
    {'q': '¿Se le puede imponer al imputado más de una restricción al mismo tiempo durante el proceso?',
     'answer': 'una o varias de las siguientes medidas cautelares'},
    {'q': '¿Qué significa que un elemento constituya un indicio con valor probatorio en la etapa inicial?',
     'answer': 'dato de prueba es la referencia al contenido de un determinado medio'},
])

tei = TEIClient(cache_dir=CACHE_DIR)
print('Embedder:', tei.model_id, '· ctx', tei.ctx)

Q_INSTRUCT = 'Instruct: Recupera el pasaje del código o la doctrina que responde la pregunta.\nQuery: '
qvecs = tei.embed([Q_INSTRUCT + q for q in golden.q])
struct_vecs = tei.embed(chunks['text_for_embedding'].tolist())
base_vecs = tei.embed(baseline['text'].tolist())

Embedder: Qwen/Qwen3-Embedding-0.6B · ctx 16384


## 3. Denso vs BM25 vs híbrido (sobre los chunks por estructura)

Relevancia = el chunk recuperado contiene la frase-respuesta. Se calcula el ranking
completo por consulta con cada recuperador y se mide `recall@{5,10,20}` y `MRR`.

In [4]:
KS = (5, 10, 20)

def rankings(texts, cvecs):
    """Para cada consulta, el orden de índices de chunk según denso / bm25 / híbrido."""
    bm25 = BM25([tokenize(t) for t in texts])
    dense, lex, hyb = [], [], []
    for qi in range(len(golden)):
        d = list(np.argsort(-(cvecs @ qvecs[qi])))
        b = rank_indices_by_score(bm25.scores(tokenize(golden.q[qi])))
        dense.append(d); lex.append(b); hyb.append(rrf([d, b]))
    return {'denso': dense, 'bm25': lex, 'híbrido': hyb}

def score(orders, texts, ks=KS):
    rows = {}
    for name, per_q in orders.items():
        hits = {k: [] for k in ks}; rr = []
        for qi, order in enumerate(per_q):
            ans = golden.answer[qi]
            rel = [ans in texts[j] for j in order]
            first = next((r for r, x in enumerate(rel, 1) if x), None)
            rr.append(1.0 / first if first else 0.0)
            for k in ks:
                hits[k].append(bool(any(rel[:k])))
        rows[name] = pd.Series({**{f'recall@{k}': float(np.mean(hits[k])) for k in ks},
                                'MRR': float(np.mean(rr))})
    return pd.DataFrame(rows).round(3)

struct_texts = chunks['text_for_embedding'].tolist()
struct_orders = rankings(struct_texts, struct_vecs)
score(struct_orders, struct_texts)

,denso,bm25,híbrido
recall@5,0.167,0.167,0.083
recall@10,0.250,0.333,0.250
recall@20,0.250,0.333,0.500
MRR,0.122,0.092,0.120


## 4. Rank del chunk correcto, por pregunta

Para ver *dónde* rescata el híbrido: rank del primer chunk relevante con cada
recuperador (∞ = no aparece).

In [5]:
def first_rank(order, ans, texts):
    for r, j in enumerate(order, 1):
        if ans in texts[j]:
            return r
    return None

rows = []
for qi in range(len(golden)):
    row = {'pregunta': golden.q[qi][:56]}
    for name, per_q in struct_orders.items():
        r = first_rank(per_q[qi], golden.answer[qi], struct_texts)
        row[name] = r if r else '∞'
    rows.append(row)
per_question = pd.DataFrame(rows)
with pd.option_context('display.max_colwidth', 62, 'display.width', 200):
    display(per_question)

,pregunta,denso,bm25,híbrido
0,¿Se puede tratar a un acusado como culpable antes de que,10,157,20
1,¿Qué grado de certeza necesita el tribunal para poder co,1,3,1
2,¿Cómo se garantiza que la evidencia recogida en la escen,27,7,8
3,"Si la policía obtiene una prueba violando derechos, ¿esa",1194,2621,1789
4,¿Qué debe ocurrir para que el fiscal logre que el proces,37,711,96
5,¿Puede la víctima llegar a un arreglo con el acusado par,310,672,487
6,¿Pueden juzgar a alguien otra vez por un delito del que,5,777,20
7,¿La defensa tiene derecho a rebatir y objetar los elemen,43,722,120
8,¿En qué condición es legítimo detener y retener físicame,127,499,248
9,¿Qué diligencias puede llevar a cabo el ministerio públi,27,950,73


## 5. ¿La estructura sigue ayudando? (estructura vs tamaño fijo, mismo recuperador)

Para no perder el hallazgo del otro notebook: se repite la mejor configuración
(híbrido) sobre el baseline de tamaño fijo y se compara con la estructura.

In [6]:
base_texts = baseline['text'].tolist()
base_orders = rankings(base_texts, base_vecs)

comparacion = pd.DataFrame({
    'estructura · híbrido': score({'h': struct_orders['híbrido']}, struct_texts)['h'],
    'tamaño fijo · híbrido': score({'h': base_orders['híbrido']}, base_texts)['h'],
    'estructura · denso': score({'h': struct_orders['denso']}, struct_texts)['h'],
}).round(3)
comparacion

,estructura · híbrido,tamaño fijo · híbrido,estructura · denso
recall@5,0.083,0.083,0.167
recall@10,0.250,0.167,0.250
recall@20,0.500,0.167,0.250
MRR,0.120,0.106,0.122


## Conclusiones

Se llenan al correr, pero la lectura esperada:

* **híbrido ≥ denso, BM25** en `recall@{10,20}` y `MRR` — el léxico rescata las
  preguntas donde el denso traía sólo el vecino temático.
* La **estructura** debe seguir por delante del tamaño fijo con el mismo recuperador.
* Siguiente palanca: **reranker** (cross-encoder) sobre el top-N del híbrido (Lab 05).

Todo el chunking viene de `shared.legal_chunking` (probado en `tests/`), así que este
notebook y el de exploración comparten exactamente la misma lógica.